In [61]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [87]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    print(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        print("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
    observed_src.drop_duplicates(subset='indicator', inplace=True)
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    print("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


Querying owner: HTOC Org


,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,legacyLink,tags.data,associatedGroups.data,hostName,dnsActive,whoisActive,source,address,url,indicator
0,6755399459078614,2025-06-25T17:45:32Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-23T12:43:17Z,3.0,76,TASK0891295,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,195.178.110.137
1,5629499559316612,2025-07-14T15:36:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-23T12:43:17Z,5.0,99,NaN,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 464075, 'name': 'Secret Blizzard', 'la...","[{'id': 5629499535000011, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,185.104.29.40
2,5265125,2025-01-23T19:51:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-23T12:42:50Z,3.0,57,CISA conducted a hunt on IoC's obtained from o...,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 459976, 'name': 'CVE-2025-283', 'lastU...","[{'id': 546888, 'dateAdded': '2025-01-23T19:51...",NaN,NaN,NaN,NaN,NaN,NaN,23.95.44.80
3,6755399460008127,2025-07-02T12:05:35Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-23T11:34:39Z,4.0,71,"Details\nIn late June 2025, Iran-aligned hackt...",...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1469320, 'name': 'INDICATOR NOTICE 251...","[{'id': 5629499544001057, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,203.81.67.22
4,6755399460008029,2025-07-02T12:05:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-23T10:42:47Z,4.0,71,"Details\nIn late June 2025, Iran-aligned hackt...",...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1469320, 'name': 'INDICATOR NOTICE 251...","[{'id': 5629499544001057, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,147.28.240.218
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1078,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83,NaN,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,NaN,https://google,NaN,google,google
1079,4512619,2024-01-26T20:41:51Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-01-28T12:55:28Z,3.0,70,"On December 28, 2023, a VA user received and r...",...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 38829, 'name': 'Phishing', 'descriptio...","[{'id': 311535, 'dateAdded': '2024-01-26T20:41...",NaN,NaN,NaN,https://shorturl.asia,NaN,shorturl.asia,shorturl.asia
1080,4491603,2023-12-21T16:13:01Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2023-12-22T11:56:27Z,3.0,89,"On December 11, 2023, VA users received a susp...",...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 38829, 'name': 'Phishing', 'descriptio...","[{'id': 292886, 'dateAdded': '2023-12-21T16:12...",NaN,NaN,NaN,https://financier.com,NaN,financier.com,financier.com
1081,4478644,2023-11-21T18:07:18Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,EmailAddress,2023-12-05T23:22:32Z,3.0,90,ACD R&F processed a malspam campaign with an I...,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 213562, 'name': 'hhs-2023112102', 'las...","[{'id': 288608, 'dateAdded': '2023-11-21T22:02...",NaN,NaN,NaN,ACD R&F,kerry@govwhitepapers.com,NaN,kerry@govwhitepapers.com


In [88]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=1)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250723.csv']
Loaded data from 1 files.


C:\Users\jaskew\AppData\Local\Temp\ipykernel_15072\3485874090.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


In [89]:
import pandas as pd

df = observed_src.copy()

df = df[df['indicator'].isin(observed_data_df['indicator'])]
# --- FULL TAGS UNNEST + PARTNERS ---

# explode tags.data
tags_exploded = (
    df[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# normalize all fields in tags.data into flat columns
tags_norm = pd.json_normalize(tags_exploded['tags.data'])
tags_norm.columns = [f"tag_{c}" for c in tags_norm.columns]

# re‑attach indicator
tags_expanded = tags_exploded.reset_index(drop=True).join(tags_norm)

# extract partners
tags_expanded['partner'] = tags_expanded['tag_name'].map(
    lambda n: n[:-len(' Splunk API')] if isinstance(n, str) and n.endswith(' Splunk API') else None
)

# aggregate each tag_* field into a list per indicator
tag_fields = list(tags_norm.columns)
tag_agg = (
    tags_expanded
      .groupby('indicator', as_index=False)
      .agg({
          **{f: list for f in tag_fields},
          'partner': lambda x: [p for p in dict.fromkeys(x) if p]
      })
      .rename(columns={'partner':'partners'})
)

# --- GROUPS VIA EXPLODE + GROUPBY ---

groups_exploded = (
    df[['indicator', 'associatedGroups.data']]
      .explode('associatedGroups.data')
      .dropna(subset=['associatedGroups.data'])
)

group_norm = pd.json_normalize(
    groups_exploded['associatedGroups.data']
).rename(columns={'id':'group_id','name':'group_name'})

groups_exploded = groups_exploded.reset_index(drop=True).join(group_norm)

group_agg = (
    groups_exploded
      .drop_duplicates(subset=['indicator','group_id','group_name'])
      .groupby('indicator', as_index=False)
      .agg({
          'group_id':   lambda ids: list(ids),
          'group_name': lambda names: list(names)
      })
      .rename(columns={'group_id':'group_ids','group_name':'group_names'})
)

# --- CORE AGGREGATION OF OTHER COLUMNS ---

first_cols = [
    'id','dateAdded','ownerId','ownerName','webLink','type','lastModified',
    'rating','confidence','description','summary','observations',
    'lastObserved','privateFlag','active','activeLocked','ip',
    'legacyLink','hostName','dnsActive','whoisActive','source',
    'address','url'
]

base_agg = (
    df
      .drop(columns=[
          'createdBy.id','createdBy.userName','createdBy.firstName','createdBy.lastName',
          'createdBy.pseudonym','createdBy.owner','xid','eventType','documentDateAdded',
          'documentType','fileSize','fileName','downVoteCount','upVoteCount','type_group',
          'webLink_group','ownerName_group','ownerId_group','dateAdded_group','id_group',
          'platforms.count','tactics.count',
      ], errors='ignore')
      .groupby('indicator', as_index=False)[
          [c for c in first_cols if c not in ['address','ip','source','url','legacyLink']]
      ]
      .first()
)

# --- MERGE EVERYTHING BACK ---

agg_df = (
    base_agg
      .merge(tag_agg,   on='indicator', how='left')
      .merge(group_agg, on='indicator', how='left')
)

def clean_list(lst):
    if not isinstance(lst, list):
        return []
    cleaned = []
    for v in lst:
        # drop any null‑like values
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass
        # drop empty strings
        if isinstance(v, str) and v == "":
            continue
        cleaned.append(v)
    return cleaned

def list_to_csv(lst):
    """
    Takes a cleaned list and returns:
      - a comma-separated string of its items, OR
      - an empty string if there are none.
    """
    if not lst:
        return ""
    return ", ".join(str(v) for v in lst)

# apply to all your formerly‑list columns
for col in ['partners','group_ids','group_names'] + tag_fields:
    agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)

display(agg_df)



,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,tag_lastUsed,tag_description,tag_techniqueId,tag_tactics.data,tag_tactics.count,tag_platforms.data,tag_platforms.count,partners,group_ids,group_names
0,101.89.174.236,5629499542017370,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-22T07:35:33Z,3.0,72,...,"2025-07-23T10:42:47Z, 2025-07-23T12:43:17Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS",,
1,103.125.189.6,5629499556005919,2025-06-30T17:15:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-21T07:36:47Z,3.0,77,...,"2025-07-23T10:42:47Z, 2025-07-23T12:43:17Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, CDC",,
2,103.149.86.208,6755399458556969,2025-06-11T14:36:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-23T07:49:10Z,3.0,74,...,"2025-07-23T10:42:47Z, 2025-07-23T12:43:17Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS",,
3,103.191.76.181,5629499536034740,2025-04-11T17:47:40Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-22T07:35:50Z,4.0,77,...,"2025-07-23T00:00:00Z, 2025-06-25T14:32:36Z, 20...",This tag is used specifically for HTOC I&W rep...,,,,,,"CMS, NIH","6755399450169906, 6755399443004631","HTOC-20250625-1031-A, Alert to Federal CISOs: ..."
4,103.203.59.0,6755399448005412,2025-05-19T11:57:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-23T07:49:08Z,3.0,71,...,"2025-07-23T10:42:47Z, 2025-07-23T12:43:17Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS",,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
723,svgarchive.com,5263222,2025-01-22T18:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-05-30T16:33:10Z,0.0,0,...,"2025-05-30T16:59:39Z, 2025-05-30T16:59:39Z, 20...",,,,,,,NIH,"6755399448004173, 546474",NIH IRT - 65313 A FYSA Traffic Notification - ...
724,uploads-ssl.webflow.com,5629499537014058,2025-04-20T17:39:55Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-23T07:49:33Z,3.0,50,...,"2025-07-23T10:42:47Z, 2025-07-15T18:14:51Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS",,
725,vtext.com,5182028,2024-12-16T18:58:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-22T07:37:03Z,3.0,48,...,"2025-07-23T10:42:47Z, 2025-07-17T13:46:36Z, 20...",Adversaries may send phishing messages to gain...,T1566,['Initial Access'],1.0,"['Linux', 'macOS', 'Windows', 'SaaS', 'Identit...",6.0,"DHA, NIH, IHS",535913,NIH Phishing Emails 11/27/2024
726,www.sthda.com,4565837,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-23T07:49:21Z,4.0,51,...,"2025-07-23T10:42:47Z, 2025-07-23T12:43:17Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS","332131, 331131","HTOC-20240423-0923-A, VA Hosts Reaching Out to..."


In [101]:
# ──────────────────────────────────────────────────────────────────────────────
# Enrich only final filtered indicators (Shodan & VirusTotalV3) and flatten
# ──────────────────────────────────────────────────────────────────────────────
import urllib.parse
import pandas as pd

COL_PATH = "data.enrichment.data"   # adjust if API changes

indicator_values = (
    agg_df["summary"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print(f"Enriching {len(indicator_values)} indicators with Shodan & VirusTotalV3...")

enriched_results = []
failed = []

for value in indicator_values:
    try:
        encoded_value = urllib.parse.quote(value, safe="")
        q = urllib.parse.urlencode({"type": ["Shodan", "VirusTotalV3"]}, doseq=True)
        enrich_url = f"/v3/indicators/{encoded_value}/enrich?{q}"

        # Build a fresh RequestObject each loop (adjust to your SDK)
        ro = RequestObject()
        ro.set_http_method("POST")
        ro.set_request_uri(enrich_url)
        ro.set_body({})

        resp = tc.api_request(ro)
        resp.raise_for_status()

        data = resp.json()
        data["summary"] = value
        enriched_results.append(data)

    except Exception as e:
        failed.append({"summary": value, "error": str(e)})

# If nothing enriched, just carry on
if not enriched_results:
    print("No enrichment data retrieved.")
    recent_tags = agg_df.copy()

else:
    # One row per summary from enrichment payload
    df_enriched = (
        pd.json_normalize(enriched_results)
          .drop_duplicates(subset="summary", keep="last")
    )

    # Merge enrichment block into base
    recent_tags = agg_df.merge(df_enriched, on="summary", how="left")

    # ── Flatten enrichment payload without creating duplicate base rows ───────
    if COL_PATH in recent_tags.columns:
        exploded = (
            recent_tags[["summary", COL_PATH]]
            .explode(COL_PATH)
            .dropna(subset=[COL_PATH])
        )

        enrich_flat = pd.json_normalize(exploded[COL_PATH]).add_prefix("enrich_")
        enrich_flat["summary"] = exploded["summary"].values

        # ---- Aggregation helpers ---------------------------------------------
        def _flatten_lists(values):
            """Flatten one level of lists in a sequence, keep dicts intact."""
            out = []
            for v in values:
                if isinstance(v, list):
                    out.extend(v)
                else:
                    out.append(v)
            return out

        def _agg_obj(series: pd.Series):
            vals = [v for v in series.dropna()]
            if not vals:
                return None
            flat = _flatten_lists(vals)
            # if everything is hashable & not dict/list, dedupe
            if all(not isinstance(v, (list, dict)) for v in flat):
                return list(pd.unique(flat))
            return flat  # keep as-is when dicts/lists present

        obj_cols = enrich_flat.select_dtypes("object").columns.difference(["summary"])
        num_cols = enrich_flat.columns.difference(obj_cols.union({"summary"}))

        agg_dict = {c: _agg_obj for c in obj_cols}
        # choose your numeric aggregation; 'max' or 'first'
        agg_dict.update({c: "max" for c in num_cols})

        enrich_wide = (
            enrich_flat
              .groupby("summary", as_index=False)
              .agg(agg_dict)
        )

        # Remove raw list col and merge flattened cols back
        recent_tags = (
            recent_tags.drop(columns=[COL_PATH], errors="ignore")
                       .drop_duplicates(subset="summary")
                       .merge(enrich_wide, on="summary", how="left")
        )

        print(f"Successfully enriched and merged {len(df_enriched)} indicators.")
    else:
        print("Enrichment column not found; nothing to flatten.")

# Optional: report/log failures
if failed:
    print(f"{len(failed)} indicators failed to enrich.")
    # Example: pd.DataFrame(failed).to_json("enrich_failures.json", orient="records")


Enriching 728 indicators with Shodan & VirusTotalV3...


Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL accentcare.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL aka.ms/o0ukef cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host arpdcresources.ca cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host bimbinlombardia.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL bmwag-rt-prod2-t.campaign.adobe.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Er

Successfully enriched and merged 696 indicators.
32 indicators failed to enrich.


In [102]:
recent_tags


,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_type,enrich_vulnerabilities,enrich_vtMaliciousCount
0,101.89.174.236,5629499542017370,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-22T07:35:33Z,3.0,72,...,[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 22, 'data': 'SSH...",[CHINANET SHANGHAI PROVINCE NETWORK],[database],"[Shodan, VirusTotal]",None,7.0
1,103.125.189.6,5629499556005919,2025-06-30T17:15:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-21T07:36:47Z,3.0,77,...,[Viet Nam],None,None,[VIETNAM POSTS AND TELECOMMUNICATIONS GROUP],"[{'transport': 'tcp', 'port': 135, 'product': ...",[Hypernet Vietnam Technology Company Limited],None,"[Shodan, VirusTotal]",None,6.0
2,103.149.86.208,6755399458556969,2025-06-11T14:36:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-23T07:49:10Z,3.0,74,...,[Viet Nam],None,None,[4S TECHNOLOGY TRADING SERVICES COMPANY LIMITED],"[{'transport': 'tcp', 'port': 22, 'product': '...",[4S TECHNOLOGY TRADING SERVICES COMPANY LIMITED],[eol-product],"[Shodan, VirusTotal]",None,5.0
3,103.191.76.181,5629499536034740,2025-04-11T17:47:40Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-22T07:35:50Z,4.0,77,...,[Malaysia],[ix-dns.com],[zen.ix-dns.com],[Jimat Technology Solution],"[{'transport': 'tcp', 'port': 21, 'product': '...",[Jimat Technology Solution],"[self-signed, starttls]","[Shodan, VirusTotal]",None,6.0
4,103.203.59.0,6755399448005412,2025-05-19T11:57:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-23T07:49:08Z,3.0,71,...,[Japan],[ipip.net],[scan-59-0.security.ipip.net],"[Beijing Tiantexin Tech. Co., Ltd.]","[{'transport': 'tcp', 'port': 22, 'product': '...","[Beijing Tiantexin Tech. Co., Ltd.]",[scanner],"[Shodan, VirusTotal]","[{'name': 'CVE-2008-3844', 'port': 22, 'descri...",3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
723,svgarchive.com,5263222,2025-01-22T18:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-05-30T16:33:10Z,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
724,uploads-ssl.webflow.com,5629499537014058,2025-04-20T17:39:55Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-23T07:49:33Z,3.0,50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
725,vtext.com,5182028,2024-12-16T18:58:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-22T07:37:03Z,3.0,48,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
726,www.sthda.com,4565837,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-23T07:49:21Z,4.0,51,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
recent_tags.drop(columns=['tag_id', 'tag_lastUsed', 'tag_lastModified', 'tag_ownerId', 
                          'tag_ownerName', 'tag_dateAdded', 'tag_description','tag_tactics.count', 
                          'tag_platform.data', 'tag_platform.count', 'data.id', 'data.dateAdded', 'data.ownerId',
                          'data.webLink', 'data.ownerName', 'data.lastModified', 'data.summary', 'data.ip',
                          'data.legacyLink','data.source', 'enrich_cloudProvider', 'enrich_cloudRegion', 'enrich_type',
                          'id'], inplace=True, errors='ignore')

recent_tags

,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,101.89.174.236,5629499542017370,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-22T07:35:33Z,3.0,72,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 22, 'data': 'SSH...",[CHINANET SHANGHAI PROVINCE NETWORK],[database],None,7.0
1,103.125.189.6,5629499556005919,2025-06-30T17:15:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-21T07:36:47Z,3.0,77,...,[Hanoi],[Viet Nam],None,None,[VIETNAM POSTS AND TELECOMMUNICATIONS GROUP],"[{'transport': 'tcp', 'port': 135, 'product': ...",[Hypernet Vietnam Technology Company Limited],None,None,6.0
2,103.149.86.208,6755399458556969,2025-06-11T14:36:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-23T07:49:10Z,3.0,74,...,[Hanoi],[Viet Nam],None,None,[4S TECHNOLOGY TRADING SERVICES COMPANY LIMITED],"[{'transport': 'tcp', 'port': 22, 'product': '...",[4S TECHNOLOGY TRADING SERVICES COMPANY LIMITED],[eol-product],None,5.0
3,103.191.76.181,5629499536034740,2025-04-11T17:47:40Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-22T07:35:50Z,4.0,77,...,[Sepang],[Malaysia],[ix-dns.com],[zen.ix-dns.com],[Jimat Technology Solution],"[{'transport': 'tcp', 'port': 21, 'product': '...",[Jimat Technology Solution],"[self-signed, starttls]",None,6.0
4,103.203.59.0,6755399448005412,2025-05-19T11:57:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-23T07:49:08Z,3.0,71,...,[Tokyo],[Japan],[ipip.net],[scan-59-0.security.ipip.net],"[Beijing Tiantexin Tech. Co., Ltd.]","[{'transport': 'tcp', 'port': 22, 'product': '...","[Beijing Tiantexin Tech. Co., Ltd.]",[scanner],"[{'name': 'CVE-2008-3844', 'port': 22, 'descri...",3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
723,svgarchive.com,5263222,2025-01-22T18:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-05-30T16:33:10Z,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
724,uploads-ssl.webflow.com,5629499537014058,2025-04-20T17:39:55Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-23T07:49:33Z,3.0,50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
725,vtext.com,5182028,2024-12-16T18:58:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-22T07:37:03Z,3.0,48,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
726,www.sthda.com,4565837,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-23T07:49:21Z,4.0,51,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
